In [1]:
import sys
import os
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from rdkit import Chem

# Ensure src is in path to import quartz
sys.path.append(os.path.abspath('src'))

from quartz.data import MoleculeData
from quartz.grids import create_grid, to_dxgrid
from quartz.dataprocessing import calculate_average, calculate_boltz_avg, calculate_pki_avg

In [2]:
# Load PDBs from directory and map to Excel Data
ligands_dir = Path('~/Documents/Major/Ligands/cocrys_ligs').expanduser()
charges_dir = Path('tests/sample_data/FXa_PDB_RESP_charges')
excel_path = Path('~/Documents/Major/Ligands/full_database_ligands.xlsx').expanduser()

# Read Excel file
print(f"Reading Excel file: {excel_path}")
fx_data = pd.read_excel(excel_path)
# Clean column names by stripping whitespace
fx_data.columns = fx_data.columns.str.strip()

pdb_files = list(ligands_dir.glob("*_out_reduced.pdb"))

mol_data_list = []

print(f"Found {len(pdb_files)} PDB files in {ligands_dir}")

for pdb_file in pdb_files:
    # Extract PDB ID from filename (e.g. 1EZQ_out_reduced.pdb -> 1EZQ)
    pdb_id = pdb_file.name.split('_')[0]
    print(f"Processing PDB ID: {pdb_id}")
    
    # Get pKi and Ki for the PDB ID
    # Assuming column is 'PDB ID' after stripping
    row = fx_data[fx_data['PDB ID'].astype(str).str.strip() == pdb_id]
    
    if row.empty:
        print(f"No FX Data found for {pdb_id}")
        continue
        
    # Handle potential duplicates or just take first
    ki = float(row['Ki (nM)'].values[0])
    # print("Ki:", ki)
    if 'pKi' in row.columns:
        pki = float(row['pKi'].values[0])
    else:
        pki = -np.log10(ki * 1e-9)  # Convert Ki in nM to pKi
    # print("pKi:", pki)

    # Load Mol from PDB
    mol = Chem.MolFromPDBFile(str(pdb_file), removeHs=False)
    if mol is None:
        print(f"Failed to load PDB: {pdb_file.name}")
        continue

    # Path to charge file
    charge_file = charges_dir / f"{pdb_id}_out_reduced.csv"
    
    if not charge_file.exists():
        print(f"Charge file not found for {pdb_id}")
        continue
        
    charges_df = pd.read_csv(charge_file)
    
    # Check atom count and charge assignment
    if len(charges_df) != mol.GetNumAtoms():
        print(f"Atom count mismatch for {pdb_id}: mol {mol.GetNumAtoms()} vs csv {len(charges_df)}")
        continue

    # Assign RESP charges
    charge_sum = 0.0
    for idx, row_charge in charges_df.iterrows():
        atom = mol.GetAtomWithIdx(idx)
        resp_charge = float(row_charge['RESP'])
        atom.SetProp("RESP", str(resp_charge))
        charge_sum += resp_charge
        
        # Verify assignment
        if abs(float(atom.GetProp("RESP")) - resp_charge) > 1e-6:
             print(f"Charge assignment failed for {pdb_id} atom {idx}")

    # print(f"Processed {pdb_id}: Total Charge = {charge_sum:.4f}")
        
    # Create MoleculeData
    md = MoleculeData(
        atomic_numbers=np.array([a.GetAtomicNum() for a in mol.GetAtoms()]),
        coordinates=mol.GetConformer().GetPositions(),
        res={},
        smiles=Chem.MolToSmiles(mol),
        rdmol=mol,
        pdb_code=pdb_id,
        filename=pdb_id,
        ki=ki,
        pki=pki
    )
    mol_data_list.append(md)

print(f"Prepared {len(mol_data_list)} MoleculeData objects")

# Generate grids
print("Generating grids...")
grids = create_grid(mol_data_list, props=["RESP"], threshold=0.0001)

# # Save to pkl
# output_file = 'grids.pkl'
# with open(output_file, 'wb') as f:
#     pickle.dump(grids, f)
    
# print(f"Saved grids to {output_file}")

# # Save to DX
# print("Saving to DX format...")
# to_dxgrid(grids)
# print("Saved DX files to grids/ directory")

Reading Excel file: /home/friedrich/Documents/Major/Ligands/full_database_ligands.xlsx
Found 112 PDB files in /home/friedrich/Documents/Major/Ligands/cocrys_ligs
Processing PDB ID: 4ZHA
Processing PDB ID: 2XBV
Processing PDB ID: 4Y76
Processing PDB ID: 2P95
Processing PDB ID: 2XBX
Processing PDB ID: 4Y71
Processing PDB ID: 4Y6D
Processing PDB ID: 4BTU
Processing PDB ID: 4BTT
Processing PDB ID: 4BTI
Processing PDB ID: 4A71
Processing PDB ID: 2XC0
Processing PDB ID: 3TK5
Processing PDB ID: 3SW2
Processing PDB ID: 3Q3K
Failed to load PDB: 3Q3K_out_reduced.pdb
Processing PDB ID: 2PHB
Processing PDB ID: 2XC4
Processing PDB ID: 3LIW
Processing PDB ID: 2Y7Z
Processing PDB ID: 3KQD
Processing PDB ID: 3KQC
Processing PDB ID: 3KQB
Processing PDB ID: 3KL6
Processing PDB ID: 3K9X
Processing PDB ID: 3HPT
Processing PDB ID: 2UWL
Processing PDB ID: 3CS7
Processing PDB ID: 3CEN
Processing PDB ID: 3ENS
Processing PDB ID: 2Y7X
Processing PDB ID: 2Y5H
Processing PDB ID: 2XC5
Processing PDB ID: 2J2U
Proce

[15:18:57] Explicit valence for atom # 21 C, 5, is greater than permitted


Processing PDB ID: 2CJI
Processing PDB ID: 2J38
Processing PDB ID: 2VH0
Processing PDB ID: 3M37
Processing PDB ID: 2VVC
Processing PDB ID: 2BOH
Processing PDB ID: 2EI8
Processing PDB ID: 2J95
Processing PDB ID: 2VH6
Processing PDB ID: 3TK6
Processing PDB ID: 2RA0
Processing PDB ID: 2Q1J
Processing PDB ID: 2PR3
Processing PDB ID: 2VVU
Processing PDB ID: 4Y7A
Processing PDB ID: 2P94
Processing PDB ID: 4Y7B
Processing PDB ID: 2P3U
Processing PDB ID: 2P3T
Processing PDB ID: 2P16
Processing PDB ID: 2JKH
Processing PDB ID: 4Y79
Processing PDB ID: 2J94
Processing PDB ID: 2VVV
Processing PDB ID: 2J4I
Processing PDB ID: 2G00
Processing PDB ID: 2VWL
Processing PDB ID: 2EI7
Processing PDB ID: 2D1J
Processing PDB ID: 2VWM
Processing PDB ID: 2BQ7
Processing PDB ID: 2BQ6
Processing PDB ID: 2VWN
Processing PDB ID: 2BMG
Processing PDB ID: 1WU1
Processing PDB ID: 1V3X
Processing PDB ID: 1QBN
Processing PDB ID: 2W3K
Processing PDB ID: 1NFX
Processing PDB ID: 1NFW
Processing PDB ID: 1MQ6
Processing PDB I

In [3]:
mol_data_list[0].rdmol.GetAtomWithIdx(0).GetProp("RESP")
mol_data_list[0].filename

'4ZHA'

In [4]:
np.min(grids[0].npGrid['RESP'])

np.float16(-0.4905)

In [5]:
avg = calculate_average(grids, props=["RESP"])
boltz_avg = calculate_boltz_avg(grids, props=["RESP"])
pki_avg = calculate_pki_avg(grids, props=["RESP"])

In [6]:
print(np.min(avg["RESP"]), np.min(boltz_avg["RESP"]), np.min(pki_avg["RESP"]))
print(np.max(avg["RESP"]), np.max(boltz_avg["RESP"]), np.max(pki_avg["RESP"]))

-0.05746505129227945 -0.09010799150208511 -0.5692498947330398
0.17128724625351233 0.1187085452153316 1.1509709535038546
